# Lezione 20 — Clustering: raggruppare memorie senza etichette

**Come si usa questo notebook.** Come sempre. Prerequisito: Lezione 19
(visualizzare gli embedding con la PCA). Li' abbiamo *guardato* se le
memorie dello stesso type si raggruppano nel grafico; oggi rendiamo quella
domanda **operativa**: un algoritmo che assegna ogni memoria a un gruppo,
senza mai vedere l'etichetta `type`, riesce a ritrovare la stessa
struttura?

**Cosa saprai fare alla fine:** applicare `KMeans`, capire perche' le
etichette di cluster non hanno un ordine o un nome intrinseco, e valutare
onestamente un clustering con l'Adjusted Rand Index invece che con
un'accuratezza che non ha senso in questo contesto.

## Parte 1 — Teoria: raggruppare senza sapere il nome del gruppo

Tutte le lezioni precedenti di questo modulo usavano il `type` come
etichetta nota (per addestrare il classificatore che produce l'embedding,
Lezioni 16-17) o come riferimento per verificare l'embedding (Lezioni
18-19). Il **clustering** e' diverso: e' *non supervisionato* — l'algoritmo
raggruppa i punti guardando solo le loro posizioni nello spazio degli
embedding, senza mai vedere `type`. Serve quando le etichette non ci sono
affatto (memorie nuove, mai categorizzate) o quando si vuole scoprire una
struttura che le etichette esistenti non catturano.

**K-Means** e' l'algoritmo piu' comune: si fissa `k` (il numero di gruppi
voluti), poi si ripete un ciclo in due passi fino a convergenza:

1. **Assegnazione** — ogni punto va al centroide piu' vicino (di solito
   distanza euclidea).
2. **Aggiornamento** — ogni centroide si sposta nella media dei punti
   assegnati a lui.

L'algoritmo minimizza l'**inerzia**: la somma delle distanze al quadrato
tra ogni punto e il centroide del suo cluster. `k` va scelto **prima**
(non lo impara da solo) — nel progetto lo fisseremo a 4, sapendo gia'
quante categorie esistono, ma in un caso reale senza etichette si
sceglie con euristiche come il metodo del gomito (guardare come cala
l'inerzia al crescere di `k` e fermarsi dove il calo rallenta).

**Un punto cruciale, facile da sbagliare**: le etichette di cluster
(`0, 1, 2, 3`) sono **arbitrarie**. `KMeans` non sa che esiste un cluster
chiamato "episodic" — assegna solo numeri, e il numero `0` di
un'esecuzione puo' corrispondere al numero `2` di un'altra. Confrontare
`etichette_cluster == etichette_vere` direttamente non ha senso (nessuna
ragione che il cluster "0" corrisponda al type "episodic" invece che a
un altro). Due strumenti onesti per confrontare comunque:

- una **tabella di contingenza** (`pd.crosstab`), che mostra quante
  memorie di ogni `type` sono finite in ogni cluster — leggibile a
  occhio, senza pretendere un numero unico;
- l'**Adjusted Rand Index** (`sklearn.metrics.adjusted_rand_score`), una
  metrica che misura l'accordo tra due raggruppamenti **indipendentemente
  dai nomi** delle etichette: `1.0` accordo perfetto, `0.0` accordo
  atteso per caso, valori negativi possibili (peggio del caso).

In [1]:
import numpy as np
from sklearn.cluster import KMeans

# Due gruppi ovvi in 2D, ben separati.
gruppo_a = np.array([[0.0, 0.0], [0.2, -0.1], [-0.1, 0.1]])
gruppo_b = np.array([[10.0, 10.0], [10.1, 9.9], [9.9, 10.2]])
punti_toy = np.vstack([gruppo_a, gruppo_b])

kmeans_toy = KMeans(n_clusters=2, random_state=42, n_init=10)
etichette_toy = kmeans_toy.fit_predict(punti_toy)
print('punti:', punti_toy.tolist())
print('etichette assegnate:', etichette_toy)
print('centroidi:', np.round(kmeans_toy.cluster_centers_, 2))

punti: [[0.0, 0.0], [0.2, -0.1], [-0.1, 0.1], [10.0, 10.0], [10.1, 9.9], [9.9, 10.2]]
etichette assegnate: [0 0 0 1 1 1]
centroidi: [[ 0.03  0.  ]
 [10.   10.03]]


Leggi l'output: i primi 3 punti (vicini tra loro, vicino all'origine) e
gli ultimi 3 (vicini tra loro, vicino a `[10, 10]`) ricevono la stessa
etichetta all'interno del proprio gruppo — ma **quale numero** (`0` o `1`)
tocca a quale gruppo dipende dall'inizializzazione casuale dei centroidi,
non da un ordine "giusto". Se rieseguissi con un seed diverso, gli stessi
due gruppi potrebbero uscire con le etichette scambiate.

## Parte 2 — Esercizio guidato

Il tuo compito: calcola l'inerzia di `KMeans(n_clusters=1)` sugli stessi
`punti_toy` (un solo centroide per tutti e 6 i punti) e confrontala con
`kmeans_toy.inertia_` (2 cluster). Cosa ti aspetti, visto quanto sono
distanti i due gruppi?

In [2]:
# Scrivi qui: KMeans(n_clusters=1, random_state=42, n_init=10) su punti_toy,
# stampa .inertia_ e confrontala con kmeans_toy.inertia_.

pass

### Soluzione spiegata

Con un solo centroide, tutti e 6 i punti contribuiscono alla stessa somma
di distanze al quadrato — e i due gruppi sono lontanissimi tra loro
(`[0,0]` contro `[10,10]`), quindi l'inerzia con 1 cluster deve essere
**molto** piu' alta di quella con 2: il singolo centroide finisce a meta'
strada, lontano da entrambi i gruppi.

In [3]:
kmeans_1 = KMeans(n_clusters=1, random_state=42, n_init=10)
kmeans_1.fit(punti_toy)
print(f'inerzia con 1 cluster: {kmeans_1.inertia_:.2f}')
print(f'inerzia con 2 cluster: {kmeans_toy.inertia_:.2f}')
assert kmeans_1.inertia_ > kmeans_toy.inertia_

inerzia con 1 cluster: 300.14
inerzia con 2 cluster: 0.13


## Parte 3 — Il progetto: Memory AI Lab, passo 20 — clustering delle memorie

Ricostruiamo l'incorporatore delle Lezioni 17-19 e applichiamo `KMeans`
con `k=4` (sappiamo che esistono 4 type, ma l'algoritmo non li vede)
direttamente sui 16 numeri dell'embedding — non sulla proiezione PCA a 2D
della Lezione 19, che era solo per il grafico: il clustering lavora sui
dati veri, non su una loro compressione visiva.

In [4]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import pandas as pd
import keras
from keras.layers import TextVectorization
from pathlib import Path
from sklearn.metrics import adjusted_rand_score

keras.utils.set_random_seed(42)

processed = Path('..') / 'datasets' / 'processed'
memorie = {n: pd.read_csv(processed / f'memory_{n}.csv') for n in ['train', 'val']}
classi = ['episodic', 'preference', 'semantic', 'unknown']
mappa = {c: i for i, c in enumerate(classi)}
testi = {n: f['text'].astype(str).to_numpy() for n, f in memorie.items()}
target = {n: f['type'].map(mappa).to_numpy() for n, f in memorie.items()}

LUNGHEZZA_SEQUENZA = 24
vettorizzatore = TextVectorization(max_tokens=300, output_mode='int',
                                   output_sequence_length=LUNGHEZZA_SEQUENZA)
vettorizzatore.adapt(testi['train'])
X_seq = {n: vettorizzatore(t).numpy() for n, t in testi.items()}

ingresso = keras.Input(shape=(LUNGHEZZA_SEQUENZA,))
parole = keras.layers.Embedding(input_dim=300, output_dim=16, mask_zero=True)(ingresso)
vettore_frase = keras.layers.GlobalAveragePooling1D(name='sentence_embedding')(parole)
nascosto = keras.layers.Dense(32, activation='relu')(vettore_frase)
nascosto = keras.layers.Dropout(0.3)(nascosto)
uscita = keras.layers.Dense(4, activation='softmax')(nascosto)

classificatore = keras.Model(ingresso, uscita)
classificatore.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
                       metrics=['accuracy'])
classificatore.fit(X_seq['train'], target['train'], epochs=300,
                   validation_data=(X_seq['val'], target['val']),
                   callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                                            restore_best_weights=True)],
                   verbose=0)

incorporatore = keras.Model(classificatore.input,
                            classificatore.get_layer('sentence_embedding').output)
vettori_val = incorporatore(X_seq['val']).numpy()
tipi_val = memorie['val']['type'].to_numpy()
print('conteggio memorie per type (validation):')
print(pd.Series(tipi_val).value_counts())

2026-07-16 21:24:53.694263: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


conteggio memorie per type (validation):
episodic      13
semantic       5
unknown        1
preference     1
Name: count, dtype: int64


In [5]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_assegnati = kmeans.fit_predict(vettori_val)

tabella = pd.crosstab(pd.Series(cluster_assegnati, name='cluster'),
                      pd.Series(tipi_val, name='type'))
print(tabella)

ari = adjusted_rand_score(tipi_val, cluster_assegnati)
print(f'\nAdjusted Rand Index: {ari:.3f}  (1.0 = accordo perfetto, 0.0 = livello del caso)')

type     episodic  preference  semantic  unknown
cluster                                         
0               4           0         0        0
1               0           0         5        1
2               9           0         0        0
3               0           1         0        0

Adjusted Rand Index: 0.555  (1.0 = accordo perfetto, 0.0 = livello del caso)


Guarda la tabella onestamente, non cercando di farla tornare. Il
validation set e' sbilanciato (13 `episodic`, 5 `semantic`, 1 `unknown`,
1 `preference` nell'esecuzione di riferimento): `KMeans` tende a produrre
cluster di dimensione simile tra loro, perche' minimizza una somma di
distanze, non conosce e non rispetta i conteggi reali delle classi. Il
risultato tipico: il cluster piu' numeroso (`episodic`) viene **spezzato
in due** cluster diversi, mentre le classi rare (`unknown`, `preference`)
finiscono mescolate dentro cluster dominati da altro invece di avere un
cluster tutto loro. L'Adjusted Rand Index intermedio (ne' vicino a `1.0`
ne' a `0.0`) riflette esattamente questo: l'algoritmo ha trovato *una*
struttura reale nello spazio degli embedding, ma non e' la stessa
partizione di `type` — non e' un fallimento del codice, e' un limite
onesto di come K-Means gestisce classi sbilanciate senza mai vederle.

## Cosa hai imparato

- Il **clustering** raggruppa punti senza etichette; **K-Means** alterna
  assegnazione al centroide piu' vicino e aggiornamento dei centroidi,
  minimizzando l'inerzia.
- Le etichette di cluster sono **numeri arbitrari**: non confrontarle mai
  direttamente con etichette vere. Una tabella di contingenza o
  l'**Adjusted Rand Index** (invariante al nome delle etichette) sono i
  modi onesti di valutare un clustering.
- Classi **sbilanciate** rompono l'assunzione implicita "un cluster per
  categoria": K-Means tende a cluster di dimensioni simili, quindi puo'
  spezzare una classe grande o fondere classi piccole con altre.
- Un ARI intermedio non e' un bug da correggere a tutti i costi: e' una
  misura onesta di quanto la struttura scoperta dall'algoritmo coincide
  con quella che ti aspettavi.

## Quiz

1. Perche' confrontare `cluster_assegnati == tipi_val` direttamente non
   ha senso, anche se il clustering fosse perfetto?
2. Cosa minimizza l'algoritmo K-Means a ogni iterazione, e con quali due
   passi alternati?
3. Il validation set ha 13 memorie `episodic` su 20 totali. Perche'
   questo sbilanciamento rende probabile che K-Means spezzi `episodic` in
   piu' cluster invece di scoprire un cluster "pulito" per ogni type?

<details>
<summary><b>Apri le risposte</b></summary>

1. Perche' le etichette di cluster (`0, 1, 2, 3`) sono numeri assegnati
   arbitrariamente dall'algoritmo, senza nessuna corrispondenza garantita
   con l'ordine o il nome delle etichette vere: anche con un clustering
   perfetto, il cluster "0" potrebbe corrispondere al type "semantic"
   invece che a "episodic" — serve una metrica invariante alla
   permutazione delle etichette (Adjusted Rand Index) o una tabella di
   contingenza.
2. Minimizza l'**inerzia** (somma delle distanze al quadrato tra ogni
   punto e il centroide del proprio cluster), alternando **assegnazione**
   (ogni punto al centroide piu' vicino) e **aggiornamento** (ogni
   centroide alla media dei punti assegnati).
3. K-Means minimizza una somma di distanze totali, non "conosce" i
   conteggi reali delle classi: con un gruppo molto piu' numeroso degli
   altri, spesso conviene all'algoritmo (in termini di inerzia) dividere
   quel gruppo grande in due cluster piuttosto che lasciare intatto un
   cluster enorme e forzarne uno minuscolo per una classe rara — il
   risultato e' esattamente lo split osservato nella tabella.
</details>

## Fonti

- scikit-learn, `KMeans`:
  https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html
- scikit-learn, *Clustering*:
  https://scikit-learn.org/stable/modules/clustering.html#k-means
- scikit-learn, `adjusted_rand_score`:
  https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html

## Prossima lezione

Il clustering raggruppa; la ricerca vera e propria (dato un testo nuovo,
trovare le memorie piu' rilevanti) e' un compito diverso, che va
**misurato** con metriche dedicate — Recall@K, Precision@K, MRR — il
tema dell'ultima lezione del modulo.